[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-13-back-translation.ipynb)


# Lecture 13: back-translation, training data from text in one language

A parallel corpus is the scarce thing; text in your language alone is far easier to find.
Back-translation turns the second into the first. Train a model in the *reverse* direction, your
language to English, use it to translate a pile of sentences in your language into English, and
add those pairs to your training data: the English side is machine-made, but the side your
model learns to *write* is real. Then retrain English to your language, and measure whether it
helped. That is Assignment 13. You write the two steps that are the method, reversing the split
and combining the data without contaminating your test set; checks tell you when each is right,
and the notebook runs the rest.

By the end you will be able to:

1. reverse a translation direction by swapping the columns of your split, and train it with
   the same configuration;
2. build a training set from back-translated pairs, say which side of each pair is real and why
   that is the side that matters, and keep every held-out sentence out of it;
3. compare the new system with your A9 system on the same test set, with BLEU and COMET, and
   explain the difference against your own prediction.

**What you need.** Your A9 files in `MyDrive/CS479/assignment8`, as Lecture 9's Part B left
them: the split, `spm.model`, `a9-settings.json`, `best-a9/` and `test-a9.hyp`. And a file of
sentences in your language, one per line, that are **not** in your train, validation or test
sets: at least as many as your training set has pairs. Put it beside them as `mono.txt`.

**This is long.** Two trainings, the second on twice the data, so plan for about three times
your A8 training. Use an A100, L4 or G4. Every step saves to Drive and can be picked up in a new
session: run the two setup cells, then the step you were on.


<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 13 (*Strategies for NMT of Low-Resource Languages*). A head start on **A13**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Setup: the GPU, Drive, your A9 files, and the configuration both trainings use.
import json
import random

import pandas as pd
import sacrebleu
import torch
from torchlingo.checkpoint import load_checkpoint
from torchlingo.colab import setup
from torchlingo.config import Config
from torchlingo.data_processing import create_dataloaders
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import translate_batch, translate_to_file
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
A8_DIR = env.drive_dir / "CS479" / "assignment8"

needed = ["train.tsv", "val.tsv", "test.tsv", "spm.model", "a9-settings.json",
          "best-a9/model_best.pt", "test-a9.hyp"]
missing = [n for n in needed if not (A8_DIR / n).exists()]
if missing:
    raise SystemExit(f"Not found in {A8_DIR}: {', '.join(missing)}. Lecture 9's Part B writes "
                     "them; finish A9 first.")

settings = json.loads((A8_DIR / "a9-settings.json").read_text(encoding="utf-8"))
NO_SPACES = settings["no_spaces"]
MAX_DECODE = settings["max_decode_length"]
TOKENIZE = "flores200" if NO_SPACES else "13a"     # the BLEU A9 was scored with
SPM = str(A8_DIR / "spm.model")
vocab = SentencePieceVocab(SPM)         # one vocabulary for both languages, so either direction

EPOCHS = 35                             # the number you trained A8 and A9 for
SEED = 479
config = Config(
    d_model=512,
    n_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    batch_size=64,
    num_workers=0,
    max_decode_length=MAX_DECODE,
)


def read_tsv(name):
    return pd.read_csv(A8_DIR / name, sep="\t", dtype=str, keep_default_na=False)


def new_model():
    torch.manual_seed(SEED)
    return SimpleTransformer(src_vocab_size=len(vocab), tgt_vocab_size=len(vocab), config=config).to(device)


def train(train_tsv, val_tsv, run_name, best_dir):
    """A9's training call on new files; resumes from its checkpoint if the session dropped."""
    random.seed(SEED)
    train_loader, val_loader, _, _ = create_dataloaders(
        A8_DIR / train_tsv, A8_DIR / val_tsv, use_bucketing=True, num_workers=0, config=config,
        use_sentencepiece=True, sp_model_path=SPM, sp_tgt_model_path=SPM,
    )
    model = new_model()
    result = train_model(model, train_loader, val_loader=val_loader, num_epochs=EPOCHS,
                         device=device, config=config, gradient_clip=1.0, save_dir=A8_DIR / best_dir,
                         log_every=50, checkpointer=TrainingCheckpointer(experiment_name=run_name))
    print(f"Epochs: {len(result.train_losses)}   final val loss {result.val_losses[-1]:.4f}")


def load_best(best_dir):
    model = new_model()
    model.load_state_dict(load_checkpoint(A8_DIR / best_dir / "model_best.pt", map_location=device)["model_state_dict"])
    return model.eval()


def translate_all(model, sentences):
    out = []
    for start in range(0, len(sentences), 64):
        out.extend(translate_batch(model, sentences[start:start + 64], vocab, vocab,
                                   decode_strategy="greedy", device=device, config=config))
    return out


train_df, val_df, test_df = read_tsv("train.tsv"), read_tsv("val.tsv"), read_tsv("test.tsv")
print(f"Your split: {len(train_df):,} train, {len(val_df):,} validation, {len(test_df):,} test pairs.")


## Step 1: the monolingual text

The sentences you back-translate must be new to every part of your split. One that is already in
your test set would put its reference into training, and the score would measure memory. The cell
removes any that are, removes repeats, and keeps as many as your training set has pairs, so the
new training set is about twice the old one.


In [ ]:
MONO_FILE = A8_DIR / "mono.txt"         # <- your sentences, one per line, in your language

lines = [s.strip() for s in MONO_FILE.read_text(encoding="utf-8").splitlines()]
in_split = set(train_df.tgt) | set(val_df.tgt) | set(test_df.tgt)
fresh = list(dict.fromkeys(s for s in lines if s and s not in in_split))
random.Random(SEED).shuffle(fresh)
mono = fresh[:len(train_df)]

print(f"{len(lines):,} lines in {MONO_FILE.name}; {len(lines) - len(fresh):,} were empty, repeated or "
      f"already in your split.")
print(f"Back-translating {len(mono):,} sentences, against {len(train_df):,} original training pairs.")
if len(mono) < len(train_df):
    print("Fewer than your training pairs: the assignment asks for at least as many. Find more, "
          "or say in the write-up why you could not.")
(A8_DIR / "mono-used.txt").write_text("\n".join(mono) + "\n", encoding="utf-8")


## Step 2: the reverse model, your language to English

**Yours to write: `reverse`.** It takes a table of pairs and returns the same pairs in the other
direction, your language as the source and English as the target. The SentencePiece model you
trained in Lecture 9 covers both languages, so that swap is the whole change; the configuration
and vocabulary stay A9's. The cell checks your function on a case whose answer is known, then on
all three files of your split, before it writes them.

The cell after it is the first of the two long trainings. If the session drops, run it again and
it resumes.


In [ ]:
def reverse(frame):
    """Return the same pairs in the other direction: your language as src, English as tgt.

    TODO: write it, in a line or two. The new src column is the old tgt, and the new tgt is the
    old src. Keep the two columns named src and tgt, in that order.
    """
    raise NotImplementedError("write reverse, then run this cell again")


# A case whose answer you know, before trusting it on your split.
toy = reverse(pd.DataFrame({"src": ["good morning"], "tgt": ["buenos días"]}))
assert list(toy.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert toy.values.tolist() == [["buenos días", "good morning"]], "reverse is not swapping yet"

for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    flipped = reverse(frame)
    assert list(flipped.src) == list(frame.tgt) and list(flipped.tgt) == list(frame.src), (
        f"reversing changed the {name} pairs, not just their direction")
    flipped.to_csv(A8_DIR / f"{name}-reversed.tsv", sep="\t", index=False)
print("Checked and written: train-reversed.tsv, val-reversed.tsv, test-reversed.tsv")


In [ ]:
# The first long training: your language to English. If the session drops, run this again.
train("train-reversed.tsv", "val-reversed.tsv", "assignment-13-reverse", "best-a13-reverse")


How good is the back-translator? Its English becomes the source side of half your new training
data, so its quality bounds what back-translation can do. The cell scores it on your test set,
reversed. Expect it to be below your English-to-X score, or above it; either is worth a sentence
in the write-up.


In [ ]:
reverse = load_best("best-a13-reverse")
reverse_test = read_tsv("test-reversed.tsv")
hyp_reverse = translate_all(reverse, list(reverse_test.src))
bleu_reverse = sacrebleu.corpus_bleu(hyp_reverse, [list(reverse_test.tgt)])   # English: default BLEU
print(f"Your language to English, on your test set: {bleu_reverse}")


## Step 3: back-translate

The reverse model translates every sentence from Step 1 into English, writing to
`mono-used.bt.en` on Drive as it goes. A large set takes a while; if the session drops, run the
cell again and it carries on from the last finished batch rather than starting over.


In [ ]:
reverse = load_best("best-a13-reverse")
mono = (A8_DIR / "mono-used.txt").read_text(encoding="utf-8").splitlines()
back = translate_to_file(reverse, mono, vocab, vocab, A8_DIR / "mono-used.bt.en",
                         decode_strategy="greedy", device=device, config=config)

for bt, real in list(zip(back, mono))[:3]:
    print(f"made by the model: {bt}\nreal:              {real}\n")


## Step 4: combine, predict, then retrain English to your language

**Yours to write: `combine`.** The new training set is your A9 training pairs, unchanged and
first, then one pair per back-translated sentence: the model's English as the source, your real
sentence as the target. Two kinds of new pair must be left out: one whose English came back
empty, and one whose English is exactly a source sentence of your validation or test set. The
second is easy to miss and is the one that matters: training on a test sentence's source makes
the test score measure memory. The cell checks your function on a known case, then runs the
library's contamination check against both held-out sets before it writes `train-a13.tsv`.

Then **write a prediction**, in `PREDICTION`: will BLEU go up or down against A9, by roughly how
much, and why? The training after it is the second long one, with A9's validation and test sets
unchanged, so the added data is the only difference.


In [ ]:
from torchlingo.diagnostics import check_contamination


def combine(original, back, mono, held_out):
    """Return the A13 training set: the original pairs, then the back-translated ones.

    TODO: write it. back[i] is the reverse model's English for mono[i], so each new pair has
    back[i] as its source and mono[i] as its target. Keep the original pairs first, unchanged,
    with the columns src and tgt. Leave out a new pair if its English is empty, or if its
    English is in held_out, the source sentences of your validation and test sets.
    """
    raise NotImplementedError("write combine, then run this cell again")


# A case whose answer you know: one good new pair, one empty, and one that is a test source.
toy = combine(pd.DataFrame({"src": ["hello"], "tgt": ["hola"]}),
              ["a dog", "", "a test sentence"], ["un perro", "vacío", "una frase"],
              held_out={"a test sentence"})
assert list(toy.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert toy.values.tolist() == [["hello", "hola"], ["a dog", "un perro"]], (
    "combine is not right yet: for this case it should return exactly the two pairs shown")

mono = (A8_DIR / "mono-used.txt").read_text(encoding="utf-8").splitlines()
back = (A8_DIR / "mono-used.bt.en").read_text(encoding="utf-8").split("\n")[:len(mono)]
held_out = set(val_df.src) | set(test_df.src)
combined = combine(train_df[["src", "tgt"]], back, mono, held_out)

assert combined.iloc[:len(train_df)].values.tolist() == train_df[["src", "tgt"]].values.tolist(), (
    "the original pairs must come first, unchanged")
for name, frame in (("val", val_df), ("test", test_df)):
    result = check_contamination(frame.src, combined.src, name=name)
    print(result)
    assert result, "a held-out source sentence is in the new training set; see combine's TODO"

combined.to_csv(A8_DIR / "train-a13.tsv", sep="\t", index=False)
print(f"\nWritten: train-a13.tsv, {len(combined):,} pairs: {len(train_df):,} original and "
      f"{len(combined) - len(train_df):,} back-translated.")


In [ ]:
# Your prediction, in one or two sentences, before training.
PREDICTION = ""
if not PREDICTION.strip():
    print("No prediction written. Write one before you train; the write-up will say it is missing.\n")

# The second long training: English to your language, on train-a13.tsv. If the session drops,
# run this again.
train("train-a13.tsv", "val.tsv", "assignment-13", "best-a13")


## Step 5: score both systems the same way

The new model translates the same test set, greedily, to `test-a13.hyp`, and both it and your A9
translations, `test-a9.hyp`, are scored with the same BLEU A9 used. Both were decoded the same way
by the same code, so the difference is the added data's doing.


In [ ]:
new = load_best("best-a13")
sources, references = list(test_df.src), list(test_df.tgt)
hyp_a13 = translate_all(new, sources)
(A8_DIR / "test-a13.hyp").write_text("\n".join(hyp_a13) + "\n", encoding="utf-8")
hyp_a9 = (A8_DIR / "test-a9.hyp").read_text(encoding="utf-8").split("\n")[:len(sources)]

bleu_a9 = sacrebleu.corpus_bleu(hyp_a9, [references], tokenize=TOKENIZE)
bleu_a13 = sacrebleu.corpus_bleu(hyp_a13, [references], tokenize=TOKENIZE)
print(f"A9:  {bleu_a9}")
print(f"A13: {bleu_a13}")

# The files COMET reads, one sentence per line, the same line numbers across all four.
(A8_DIR / "test.src.txt").write_text("\n".join(sources) + "\n", encoding="utf-8")
(A8_DIR / "test.ref.txt").write_text("\n".join(references) + "\n", encoding="utf-8")


## Step 6: COMET

The same reference-based COMET model you installed for A10, on both systems at once. It needs no
Hugging Face token. It is the last cell to run, because installing COMET can change packages the
cells above rely on; if it gives trouble, run it in a fresh session on its own, after the setup
cells.


In [ ]:
%pip install --quiet unbabel-comet
!comet-score -s "{A8_DIR}/test.src.txt" -r "{A8_DIR}/test.ref.txt" -t "{A8_DIR}/test-a9.hyp" "{A8_DIR}/test-a13.hyp" --gpus 1 --quiet --only_system


## Step 7: the write-up

The cell below prints the numbers in the order the write-up asks for, and three test sentences
the two systems translated differently. For each of the three, add Google Translate's English
for both outputs, so a reader without your language can judge them. Add the two COMET system
scores from Step 6.

**Upload** to your folder in the Assignments shared folder: `train-a13.tsv` (the training set),
`val.tsv` and `test.tsv` (unchanged from A9), and `test-a13.hyp` (the new system's translations).


In [ ]:
combined = read_tsv("train-a13.tsv")
which = "spBLEU (flores200)" if NO_SPACES else "SacreBLEU, default tokenizer"

print("=== For the A13 write-up ===")
print(f"Original training pairs: {len(train_df):,};  back-translated and added: "
      f"{len(combined) - len(train_df):,};  total {len(combined):,}")
print(f"A9 BLEU: {bleu_a9.score:.2f}    A13 BLEU: {bleu_a13.score:.2f}    both greedy, both {which}")
print(f"Back-translator (your language to English) BLEU: {bleu_reverse.score:.2f}")
print(f"Configuration: A9's, unchanged, in both directions ({EPOCHS} epochs)")
print(f"Prediction, written before training: {PREDICTION.strip() or '(none written)'}")
print()
print("=== Three sentences the two systems translated differently ===")
shown = 0
for s, r, h9, h13 in zip(sources, references, hyp_a9, hyp_a13):
    if h9 != h13:
        print(f"\nSRC {s}\nREF {r}\nA9  {h9}\nA13 {h13}")
        shown += 1
        if shown == 3:
            break
